# 📊 Project FORESIGHT – Notebook 01: Exploratory Data Analysis
**NorthBay Living | Demand & Inventory Intelligence**

This notebook performs comprehensive EDA on the Online Retail II dataset and the 4 FORESIGHT processed tables.

---
## Objectives
1. Understand raw dataset characteristics
2. Analyse processed FORESIGHT tables
3. Revenue and sales trends
4. Product and category analysis
5. Seasonality patterns
6. Missing values and data quality
7. Business insights


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Dark theme
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='muted')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA = Path('../data/processed')
print('Environment ready ✓')

## 1. Load Processed Tables

In [ ]:
sales    = pd.read_csv(DATA / 'sales_daily.csv', parse_dates=['date'])
sku_m    = pd.read_csv(DATA / 'sku_master.csv', parse_dates=['first_sale_date','last_sale_date'])
calendar = pd.read_csv(DATA / 'calendar.csv')
inv      = pd.read_csv(DATA / 'inventory_snapshots.csv', parse_dates=['date'])

print(f'sales_daily       : {sales.shape}')
print(f'sku_master        : {sku_m.shape}')
print(f'calendar          : {calendar.shape}')
print(f'inventory_snapshots: {inv.shape}')

## 2. Missing Value Report

In [ ]:
def missing_report(df, name):
    null_df = df.isnull().sum().reset_index()
    null_df.columns = ['column', 'null_count']
    null_df['null_pct'] = (null_df['null_count'] / len(df) * 100).round(2)
    null_df['dtype'] = null_df['column'].apply(lambda c: str(df[c].dtype))
    null_df = null_df.sort_values('null_count', ascending=False)
    print(f'\n=== {name} ({len(df):,} rows) ==='); print(null_df.to_string(index=False))
    return null_df

for df, name in [(sales,'sales_daily'),(sku_m,'sku_master'),(calendar,'calendar'),(inv,'inventory')]:
    missing_report(df, name)

## 3. Revenue Trend Analysis

In [ ]:
daily_rev = sales.groupby('date')['revenue'].sum().reset_index()
daily_rev['ma7']  = daily_rev['revenue'].rolling(7, min_periods=1).mean()
daily_rev['ma28'] = daily_rev['revenue'].rolling(28, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_rev['date'], y=daily_rev['revenue'],
    name='Daily Revenue', line=dict(color='#6366f1', width=1.5), opacity=0.6,
    fill='tozeroy', fillcolor='rgba(99,102,241,0.08)'))
fig.add_trace(go.Scatter(x=daily_rev['date'], y=daily_rev['ma7'],
    name='7-Day MA', line=dict(color='#06b6d4', width=2.5)))
fig.add_trace(go.Scatter(x=daily_rev['date'], y=daily_rev['ma28'],
    name='28-Day MA', line=dict(color='#f59e0b', width=2.5, dash='dash')))
fig.update_layout(title='Revenue Trend (Daily + Moving Averages)',
    paper_bgcolor='#1e293b', plot_bgcolor='#1e293b',
    font=dict(color='#94a3b8'), height=420)
fig.show()

## 4. Category Analysis

In [ ]:
sku_rev = sales.groupby('stock_code')['revenue'].sum().reset_index()
sku_rev = sku_rev.merge(sku_m[['stock_code','category','description','abc_class']], on='stock_code')

cat_rev = sku_rev.groupby('category')['revenue'].sum().reset_index().sort_values('revenue', ascending=False)

fig = px.bar(cat_rev, x='category', y='revenue', color='revenue',
    color_continuous_scale=['#6366f1','#06b6d4'],
    title='Revenue by Category')
fig.update_layout(paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'))
fig.show()

print(f'\nTop 5 categories by revenue:')
print(cat_rev.head())

## 5. Top Selling Products

In [ ]:
top20 = sku_rev.nlargest(20, 'revenue').sort_values('revenue')

fig = go.Figure(go.Bar(
    x=top20['revenue'],
    y=top20['description'].str[:40].fillna(top20['stock_code']),
    orientation='h',
    marker=dict(color=top20['revenue'], colorscale=[[0,'#6366f1'],[1,'#06b6d4']]),
    text=top20['revenue'].apply(lambda v: f'£{v:,.0f}'),
    textposition='outside', textfont=dict(color='white')
))
fig.update_layout(title='Top 20 Products by Revenue',
    height=600, paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'))
fig.show()

## 6. ABC Analysis

In [ ]:
abc_stats = sku_m.groupby('abc_class').agg(
    sku_count=('stock_code','count'),
    total_revenue=('total_revenue','sum'),
    avg_price=('avg_unit_price','mean')
).reset_index()
abc_stats['revenue_pct'] = (abc_stats['total_revenue'] / abc_stats['total_revenue'].sum() * 100).round(1)
abc_stats['sku_pct'] = (abc_stats['sku_count'] / abc_stats['sku_count'].sum() * 100).round(1)
print(abc_stats.to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Revenue by ABC Class','SKU Count by ABC Class'],
    specs=[[{'type':'pie'},{'type':'pie'}]])
fig.add_trace(go.Pie(labels=abc_stats['abc_class'], values=abc_stats['total_revenue'],
    hole=0.5, marker_colors=['#6366f1','#06b6d4','#f59e0b'],
    textinfo='percent+label', textfont_color='white'), row=1, col=1)
fig.add_trace(go.Pie(labels=abc_stats['abc_class'], values=abc_stats['sku_count'],
    hole=0.5, marker_colors=['#6366f1','#06b6d4','#f59e0b'],
    textinfo='percent+label', textfont_color='white'), row=1, col=2)
fig.update_layout(paper_bgcolor='#1e293b', font=dict(color='#94a3b8'), height=380)
fig.show()

## 7. Seasonality Analysis

In [ ]:
s = sales.copy()
s['month'] = s['date'].dt.month_name().str[:3]
s['dow']   = s['date'].dt.day_name().str[:3]

pivot = s.groupby(['month','dow'])['quantity'].mean().unstack(fill_value=0)
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dow_order   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
pivot = pivot.reindex([m for m in month_order if m in pivot.index])
pivot = pivot.reindex(columns=[d for d in dow_order if d in pivot.columns])

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
    colorscale=[[0,'#0f172a'],[0.5,'#6366f1'],[1,'#06b6d4']],
    text=pivot.values.round(1), texttemplate='%{text}'
))
fig.update_layout(title='Seasonality Heatmap: Avg Daily Units (Month × Weekday)',
    paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'), height=420)
fig.show()

## 8. Dead Stock Analysis

In [ ]:
latest_inv = inv.sort_values('date').groupby('stock_code').last().reset_index()
dead_skus  = latest_inv[latest_inv['coverage_days'] > 365]
slow_skus  = latest_inv[(latest_inv['coverage_days'] > 90) & (latest_inv['coverage_days'] <= 365)]

print(f'Dead stock SKUs (>365d): {len(dead_skus):,}')
print(f'Slow moving SKUs (90-365d): {len(slow_skus):,}')
print(f'Active SKUs (<90d): {(latest_inv["coverage_days"] <= 90).sum():,}')
print(f'\nDead stock value: £{(dead_skus["inventory_value"]).sum():,.2f}')

fig = go.Figure(go.Histogram(x=latest_inv['coverage_days'].clip(upper=365), nbinsx=40,
    marker_color='#6366f1', opacity=0.8))
fig.add_vline(x=7, line_dash='dash', line_color='#ef4444', annotation_text='Critical')
fig.add_vline(x=90, line_dash='dash', line_color='#f59e0b', annotation_text='Overstock')
fig.update_layout(title='Inventory Coverage Distribution',
    paper_bgcolor='#1e293b', plot_bgcolor='#1e293b', font=dict(color='#94a3b8'), height=380)
fig.show()

## 9. Business Insights Summary

In [ ]:
print('=' * 60)
print('BUSINESS INSIGHTS SUMMARY')
print('=' * 60)
print(f'Total SKUs          : {sku_m["stock_code"].nunique():,}')
print(f'Total Revenue       : £{sales["revenue"].sum():,.2f}')
print(f'Date Range          : {sales["date"].min().date()} → {sales["date"].max().date()}')
print(f'Avg Daily Revenue   : £{sales.groupby("date")["revenue"].sum().mean():,.2f}')
print(f'Peak Revenue Day    : {sales.groupby("date")["revenue"].sum().idxmax().date()}')
print(f'Promotion Days %    : {sales["is_promotion"].mean()*100:.1f}%')
print(f'ABC-A Revenue Share : {sku_rev[sku_rev["abc_class"]=="A"]["revenue"].sum()/sku_rev["revenue"].sum()*100:.1f}%')
print(f'Categories          : {sku_m["category"].nunique()}')
print(f'Avg Lead Time       : {sku_m["lead_time_days"].mean():.1f} days')
print('=' * 60)